# Compile and evaluate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/06_compile_and_evaluate.ipynb)

From a `PlanningProblem` and a controller to a score: `problem.compile(controller)` builds a `ProblemEvaluator`, `pe.evaluate()` returns an `Evaluation` record (traceable under `jax.grad` and batched across Monte Carlo trials), `compare` benchmarks multiple controllers on paired draws, and `sys.compile()` gives the compiled evaluator underneath.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `ProblemEvaluator`: a controller compiled on a problem

`pe = problem.compile(controller, dt=None, *, solver="rk4", substeps=None, backend="auto", episode_length=None)` builds a `ProblemEvaluator`: it closes `controller` on the problem's plant (`controller @ plant` for a feedback block, `controller >> plant` for a source with no input), compiles the loop once, and scores its runs by the problem's cost and sets.

| Argument | Meaning |
| --- | --- |
| `controller` | A controller block, a source block, a `PlanningSolution` (its controller), or a built loop with a block `plant` |
| `dt` | Sample period of the run. A sampled controller (`pid % 0.01`) runs on its own clock, `dt` omitted. A continuous loop is integrated and recorded on `dt`, or on an automatic step when omitted |
| `solver`, `substeps` | Integration formula (`"rk4"`, `"euler"`) and steps per period |
| `backend` | `"auto"` (JAX when it is installed and the whole score traces), `"jax"`, or `"numpy"` |
| `episode_length` | Length in seconds of the run of an infinite-horizon problem (10 s when omitted, with a warning) |

| Call | Returns | Meaning |
| --- | --- | --- |
| `pe.problem`, `pe.loop` | `PlanningProblem`, `DiagramSystem` or `SampledDiagramSystem` | The task and the closed loop that runs it |
| `pe.backend`, `pe.dt`, `pe.n_steps`, `pe.horizon` | `str`, `float`, `int`, `float` | The backend chosen, the period, the number of periods, and the run's length |
| `pe.evaluator` | compiled evaluator | The compiled loop (section 6) |
| `pe.params` | `dict` | The run's numbers as one tree, a copy: the loop's blocks by id (a stateful block other than the plant with its start as the leaf `"x0"`), the problem's leaves (`"cost"`, the sets', `"x0"`, `"t0"`, `"discount_rate"`), and each free input of the loop at its nominal value |
| `pe.evaluate(params=None, key=0, n_trials=None, *, trajectories=False)` | `Evaluation` | One run of a deterministic problem, or `n_trials` draws of a stochastic one from `key`; `params` is `None` or the complete tree of `pe.params`, edited |
| `pe.trajectory(params=None, key=None)` | `Trajectory` | One run of the loop with every block output recorded; `key=None` is the nominal run. The plant's own run is `traj.of("plant")` |
| `pe.program(wrt, bounds=None, key=0, n_trials=None, *, params=None)` | `MathematicalProgram` | The tuning program over the leaves named by `wrt` ([`07_program`](07_program.ipynb)) |
| `pe.pack(params, wrt)` | `ndarray (n_z,)` | The leaves named by `wrt`, flattened into one vector `z` |
| `pe.unpack(z, wrt, params=None)` | `dict` | `params` (default `pe.params`) with the leaves named by `wrt` read from `z` |
| `pe.live_plot(wrt, *, params=None, signals=("x", "u"), every=1, pause=0.01, renderer="matplotlib")` | callback `(z, J, t)` | A progress callback for `Optimizer.solve` that redraws the closed-loop run |

Two functions of `minilink.planning` score without holding the evaluator:

- `evaluate(problem, controller, dt=None, *, solver="rk4", substeps=None, n_trials=None, key=0, trajectories=False, backend="auto", episode_length=None)` is `problem.compile(...).evaluate(...)`;
- `evaluate_trajectory(problem, traj)` scores a `Trajectory` already held: the plant's own run, or a loop's run with its `plant` role.

In [ ]:
import numpy as np

from minilink import PID, BallSet, InvertedPendulum, PlanningProblem, QuadraticCost

plant = InvertedPendulum()
Q = np.diag([10.0, 1.0])
R = 0.1 * np.eye(1)
cost = QuadraticCost.from_system(plant, Q=Q, R=R)
problem = PlanningProblem(
    plant,
    cost=cost,
    x0=np.array([0.4, 0.0]),
    horizon=4.0,
    Xf=BallSet(np.zeros(2), 0.05),
)

pd = PID(Kp=20.0, Kd=4.0)
pe = problem.compile(pd % 0.01)
pe.backend, pe.dt, pe.n_steps, list(pe.params)

In [ ]:
report = pe.evaluate()
print(report)

In [ ]:
traj = pe.trajectory()
plant_traj = traj.of("plant")
traj.signal_names, plant_traj.x.shape, plant_traj.u.shape

In [ ]:
pe.loop.plot_trajectory(traj);

## 2. `Evaluation`: one row per run, and their statistics

An `Evaluation` is a frozen record with one entry per run ($K$ runs; $K = 1$ on a deterministic problem). It is a JAX pytree: traced inside a trace, plain arrays outside.

| Field / call | Type | Meaning |
| --- | --- | --- |
| `ev.raw_cost` | `array (K,)` | $\int e^{-\rho (t - t_0)} g\, dt + e^{-\rho T} h(x(t_f))$; `+inf` for a run that is not finite |
| `ev.J` | `array (K,)` | The raw cost, plus the problem's failure cost when the run failed or missed `Xf` and `penalize_failure` is on |
| `ev.feasibility_margin` | `array (K,)` | The least margin of `x` to `X` and of `u` to `U` along the run |
| `ev.failed` | boolean `array (K,)` | A run that is not finite, or with a negative feasibility margin |
| `ev.reachability_margin` | `array (K,)` | The margin of $x(t_f)$ to `Xf` (`+inf` without `Xf`) |
| `ev.reached` | boolean `array (K,)` | `reachability_margin >= 0` |
| `ev.x0` | `array (K, n)` | The starts |
| `ev.trajectories` | `list[Trajectory] \| None` | The runs, when asked with `trajectories=True` |
| `ev.criterion` | `str` or `("cvar", alpha)` | The problem's criterion |
| `ev.n_trials` | `int` | The number of runs $K$ |
| `ev.objective` | scalar | The criterion over `J` (the mean, the worst, or a CVaR): the one number a program minimizes |
| `ev.mean`, `ev.std`, `ev.worst` | scalar | Mean, standard deviation and largest value of `J` |
| `ev.cvar(alpha)` | scalar | Mean of the worst $\lceil \alpha K \rceil$ values of `J` |
| `ev.feasibility_rate`, `ev.reachability_rate`, `ev.success_rate` | scalar in $[0, 1]$ | Fraction of the runs that did not fail, that ended in `Xf`, and both |

In [ ]:
(
    report.raw_cost,
    report.J,
    report.feasibility_margin,
    report.failed,
    report.reachability_margin,
    report.reached,
    report.x0,
)

In [ ]:
# Other numbers without a recompile: a copy of the evaluator's tree, edited
heavy = pe.params
heavy["plant"]["m"] = 1.2
heavier = pe.evaluate(heavy)
report.objective, heavier.objective

## 3. Differentiating the objective

On the JAX backend the whole evaluation traces, so `jax.grad` differentiates `pe.evaluate(params).objective` with respect to every leaf of `pe.params` in one backward pass:

In [ ]:
import jax

dJ = jax.grad(lambda params: pe.evaluate(params).objective)(pe.params)
dJ["controller"]["Kp"], dJ["controller"]["Kd"], dJ["plant"]["m"]

## 4. Monte Carlo evaluation of a stochastic problem

`n_trials=200, key=1` draws 200 starts and parameter realizations and runs the batch in one vectorized call:

In [ ]:
from minilink import Uniform

random_problem = problem.replace(
    x0=Uniform([0.2, -0.2], [0.6, 0.2]),
    distributions={"plant": {"m": Uniform([0.8], [1.2])}},
)
monte_carlo = random_problem.compile(pd % 0.01).evaluate(n_trials=200, key=1)
print(monte_carlo)

In [ ]:
(
    monte_carlo.n_trials,
    monte_carlo.mean,
    monte_carlo.std,
    monte_carlo.worst,
    monte_carlo.cvar(0.1),
    monte_carlo.success_rate,
)

## 5. `compare` and `Comparison`: controllers side by side

`compare(*args, dt=None, n_trials=None, key=0, solver="rk4", substeps=None, backend="auto", episode_length=None, **entries)` has two forms:

- `compare(Name=solution, ...)` names `PlanningSolution`s side by side, read as they are;
- `compare(problem, Name=entry, ...)`, the problem first, also scores every entry by `evaluate` on the same draws (one `key`), so the trials are paired. An entry is a `PlanningSolution`, a controller block, or an `Evaluation` already computed; `dt` and the other keywords are those of `evaluate`.

It returns a `Comparison`:

| Call | Returns | Meaning |
| --- | --- | --- |
| `comp.solutions` | `dict[str, PlanningSolution]` | The entries by name (a controller entry is wrapped as a solution) |
| `comp.evaluations` | `dict[str, Evaluation] \| None` | Each scored entry's `Evaluation`; `None` when no problem was given |
| `comp[name]`, `len(comp)`, `comp.items()` | | The solutions read as a mapping |
| `print(comp)` | | The table: one row per entry, its success, record and evaluation |
| `comp.difference(a, b)` | `ndarray (K,)` | The paired differences $J_a - J_b$, one per trial |
| `comp.plot_trajectory(*, signals=("x", "u"), show=True)` | `PlotResult` | The plans and the controllers' runs overlaid |
| `comp.plot_control_law(*, x_axis=None, y_axis=None, u_axis=0, bounds=None, grid_shape=(101, 101), t=0.0, show=True)` | `PlotResult` | One panel per feedback law, one colour scale |
| `comp.plot_cost_to_go(*, x_axis=0, y_axis=1, anchor=None, bounds=None, grid_shape=(51, 51), t=0.0, jmax=None, show=True)` | `PlotResult` | One panel per cost-to-go field, one colour scale ([`08_planners`](08_planners.ipynb)) |

In [ ]:
from minilink import compare, lqr

comparison = compare(
    random_problem,
    LQR=lqr(plant, Q, R) % 0.01,
    PD=pd % 0.01,
    PID=PID(Kp=20.0, Ki=10.0, Kd=4.0) % 0.01,
    n_trials=50,
    key=1,
)
print(comparison)

In [ ]:
diff_pd_lqr = comparison.difference("PD", "LQR")
diff_pd_lqr.shape, diff_pd_lqr.mean()

## 6. The compiled evaluators: `sys.compile()`

`sys.compile(*, backend="numpy", verbose=False)` lowers a system to array functions with the object dispatch removed: on NumPy plain functions, on JAX jitted ones. There is one evaluator class per kind of system (`minilink.core.compile.evaluators`):

| System | Evaluator | Time argument |
| --- | --- | --- |
| `DynamicSystem` | `DynamicEvaluator` | `t`, seconds |
| `DiagramSystem` | `DiagramEvaluator` | `t` |
| static `System` | `StaticEvaluator` | `t` |
| `StepSystem`, `SampledSystem` | `StepEvaluator` | the tick `k` |
| `StepDiagramSystem` | `StepDiagramEvaluator` | `k` |
| `SampledDiagramSystem` | `SampledDiagramEvaluator` | the base tick `k` |

Every evaluator holds `ce.n`, `ce.m`, `ce.p`, `ce.backend`, and `ce.params`, a copy of the numbers taken at the compile: `params=None` in the calls below runs on them, and a complete tree, edited, runs at other numbers without a recompile.

A continuous evaluator (`DynamicEvaluator`, `DiagramEvaluator`):

| Call | Returns | Meaning |
| --- | --- | --- |
| `ce.f(x, u, t=0.0, params=None)` | `array (n,)` | $\dot x$ |
| `ce.h(x, u=None, t=0.0, params=None)` | `array (p,)` | The `"y"` port; `u=None` holds the nominal input |
| `ce.outputs(x, u, t=0.0, params=None)` | `dict[str, array]` | Every boundary output port |
| `ce.step(x, u, t, params=None, *, dt, solver="rk4", substeps=1)` | `array (n,)` | The state one period `dt` later, `u` held |
| `ce.rollout(x0, u_seq, t0, dt, params=None, *, solver="rk4", substeps=1, hold="zoh", n_steps=None, record=())` | `array (N + 1, n)` | The step model run over the rows of `u_seq` `(N, m)`; `u_seq=None` holds the nominal input for `n_steps` periods; `hold="linear"` reads `u_seq` as `N + 1` knots. With `record` (a diagram's signal names) it returns `(x_samples, records)` |
| `ce.rollout_batch(x0s, u_sequences=None, *, t0=0.0, dt, n_steps=None, params=None)` | `array (B, N + 1, n)` | A family of rollouts in one `vmap` (JAX): `x0s` `(B, n)`; a `params` leaf with a leading axis of size `B` is a family of parameters |
| `ce.jacobian(of, wrt, x, u, t=0.0, params=None, *, eps=1e-6)` | `array` (a `dict` for `wrt="params"`) | $\partial(\text{of})/\partial(\text{wrt})$ at the point, as `sys.jacobian`: `jax.jacfwd` on JAX, central differences on NumPy. The derivative is built once per `(of, wrt)`, so a loop takes `partial(ce.jacobian, "f", "x")` as its function |
| `diagram_ce.signals(x, u, t=0.0, params=None)` | `array` | A `DiagramEvaluator`'s internal signal buffer: every block output, stacked |

A discrete evaluator (`StepEvaluator`, `StepDiagramEvaluator`, `SampledDiagramEvaluator`) has the same `h`, `outputs` and `jacobian` on the tick `k`, and:

| Call | Returns | Meaning |
| --- | --- | --- |
| `de.step(x, u, k=0, params=None, *, t0=0.0)` | `array (n,)` | $x_{k+1}$; `t0` is the origin of a clock |
| `de.outputs(x, u, k=0, params=None, *, t0=0.0)` | `dict[str, array]` | Every boundary output port at tick `k` |
| `de.rollout(x0, u_seq=None, params=None, *, n_steps=None, t0=0.0)` | `array (N + 1, n)` | The steps run over the rows of `u_seq` |
| `de.rollout_dense(x0, u_seq=None, params=None, *, n_steps=None, t0=0.0)` | `array` | The same run with a continuous source's substeps kept (a `SampledDiagramEvaluator`'s also takes `record=()`) |
| `de.rollout_batch(x0s, u_sequences=None, *, n_steps=None, params=None, t0=0.0)` | `array (B, N + 1, n)` | A family of step runs in one `vmap` (JAX) |

In [ ]:
evaluator = plant.compile(backend="jax")

x = np.array([0.4, 0.0])
u = np.array([0.0])
(
    type(evaluator).__name__,
    evaluator.f(x, u),
    evaluator.outputs(x, u),
    evaluator.step(x, u, 0.0, dt=0.01),
)

In [ ]:
from functools import partial

# The derivative as a function of the point
df_dx = partial(evaluator.jacobian, "f", "x")
df_dx(x, u)

In [ ]:
u_seq = np.zeros((100, 1))
x_path = evaluator.rollout(x, u_seq, 0.0, 0.01)

x0s = np.array([[0.1, 0.0], [0.2, 0.0], [0.3, 0.0]])
x_paths = evaluator.rollout_batch(x0s, dt=0.01, n_steps=100)
x_path.shape, x_paths.shape

In [ ]:
# A family of masses, one per member of the batch
masses = evaluator.params
masses["m"] = np.array([0.5, 1.0, 2.0])
evaluator.rollout_batch(x0s, dt=0.01, n_steps=100, params=masses)[:, -1, 0]

In [ ]:
step_evaluator = plant.discretize(0.01).compile()
(
    type(step_evaluator).__name__,
    step_evaluator.step(x, u, k=0),
    step_evaluator.rollout(x, n_steps=100).shape,
)

Next, [`07_program`](07_program.ipynb): `MathematicalProgram`, `Optimizer`, and `OptimizationResult`.